In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import librosa
import os
from pathlib import Path

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
!7z x '/content/drive/MyDrive/dataset/base_dataset.zip' -o'/content/drive/MyDrive/dataset/project-listen/'

In [ ]:
os.chdir('/content/drive/MyDrive/')

In [ ]:
file_path = "./dataset/project-listen/dataset/car/acceleration/car_acceleration_1.wav"
y, sr = librosa.load(file_path, sr=None)
mel_spectrogram = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=256, fmax=16384)
mel_spectrogram_db = librosa.power_to_db(mel_spectrogram, ref=np.max)

In [ ]:
print(y, "|", sr)

In [ ]:
plt.figure(figsize=(10, 4))
librosa.display.specshow(mel_spectrogram_db, sr=sr, x_axis='time', y_axis='mel', fmax=16000)
plt.colorbar(format='%+2.0f dB')
plt.title('Mel Spectrogram')
plt.show()

In [ ]:
target_shape = mel_spectrogram_db.shape
print(target_shape)

In [ ]:
source_dir = Path("./dataset/project-listen/classes")
destination_dir = Path("./dataset/spectograms")

In [ ]:
def copy_directory_structure(source_dir, destination_dir):
  for dirpath, _, _ in os.walk(source_dir):
    relative_path = os.path.relpath(dirpath, source_dir)
    replica_dir = os.path.join(destination_dir, relative_path)
    os.makedirs(replica_dir, exist_ok=True)

In [ ]:
copy_directory_structure(source_dir, destination_dir)

In [ ]:
def truncate_spectrogram(spectrogram, target_shape) -> np.ndarray:
    """
    Приводит спектрограмму к определенному размеру.
    (Чтобы скармливать нейросети одинаковые по размеру массивы)

    Если спектрограмма длиннее целевого размера - обрезает справа.
    Если короче - дополняет нулями справа.

    Parameters:
    spectrogram : Исходная спектрограмма
    target_shape : Целевой размер для обрезки
    """
    if spectrogram.shape[1] > target_shape[1]:
        return spectrogram[:, :target_shape[1]]
    elif spectrogram.shape[1] < target_shape[1]:
        padding = target_shape[1] - spectrogram.shape[1]
        return np.pad(spectrogram, ((0, 0), (0, padding)), mode='constant')
    return spectrogram

In [ ]:
def convert_audio_to_spectograms(source_dir, destination_dir, target_shape) -> None:
  """
  Конвертирует аудиофайлы в спектрограммы и сохраняет их как NumPy массивы в
  определенную папку.

  Функция преобразует каждый аудиофайл в спектрограмму, обрезает или дополняет
  до фиксированного размера target_shape и сохраняет в формате .npy

  Пример target_shape: (256, 173)

  Parameters:
      source_dir: Путь к корневой директории с исходными аудиофайлами
      destination_dir : Путь к корневой директории для сохранения спектрограмм
  """
  for category in os.listdir(source_dir):
    category_path = os.path.join(source_dir, category)
    for target in os.listdir(category_path):
      target_path = os.path.join(category_path, target)
      destination_target_path = os.path.join(destination_dir, category, target)
      for fname in os.listdir(target_path):
        source_file_path = os.path.join(target_path, fname)

        y, sr = librosa.load(source_file_path, sr=None)
        mel_spectrogram = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=256, fmax=16384)
        mel_spectrogram_db = librosa.power_to_db(mel_spectrogram, ref=np.max)
        processed_spectrogram = truncate_spectrogram(mel_spectrogram_db, target_shape)

        if processed_spectrogram.shape == target_shape:
          destination_file_path = os.path.join(destination_target_path, f"{os.path.splitext(fname)[0]}.npy")
          np.save(destination_file_path, processed_spectrogram)
        else:
          print(f"Warning: Processed spectrogram for {fname}"
          f"has invalid shape {processed_spectrogram.shape}")

In [ ]:
convert_audio_to_spectograms(source_dir, destination_dir, target_shape)

In [ ]:
category_mapping = {
    0: "car",
    1: "emv",
    2: "motorcycle",
    3: "tram",
    4: "truck"
}

target_mapping = {
    0: "acceleration",
    1: "bell",
    2: "braking",
    3: "horn",
    4: "idling",
    5: "passing",
    6: "siren"
}

In [ ]:
def hierarchical_one_hot_encoding(categories, targets):
    """
    One-hot кодирование таргета для категории трафика и характера
    движения этого трафика

    Parameters:
      categories - категории трафика на звуке
      targets - характер движения трафика на звуке
    """
    category_labels = LabelEncoder().fit_transform(categories)
    category_one_hot = F.one_hot(torch.from_numpy(category_labels)).float()

    target_labels = LabelEncoder().fit_transform(targets)
    target_one_hot = F.one_hot(torch.from_numpy(target_labels)).float()

    return category_one_hot, target_one_hot

In [ ]:
def load_data(destination_dir):
    """
    Загружает преобразованные спектрограммы из .npy файлов и создает
    набор данных с метками для обучения модели.

    Parameters:
      destination_dir: путь до .npy фвйла, где содержатся спектограммы
    """

    data = []             # для загрузки спектрограмм
    labels_category = []  # для меток категорий
    labels_target = []    # для таргет меток

    for category in os.listdir(destination_dir):
        category_path = os.path.join(destination_dir, category)
        for target in os.listdir(category_path):
            target_path = os.path.join(category_path, target)
            for fname in os.listdir(target_path):
              spectrogram_path = os.path.join(target_path, fname)
              spectrogram = np.load(spectrogram_path, allow_pickle=True)
              data.append(spectrogram)
              labels_category.append(category)
              labels_target.append(target)

    data = np.array(data)
    labels_category = np.array(labels_category)
    labels_target = np.array(labels_target)

    category_one_hot, target_one_hot = hierarchical_one_hot_encoding(labels_category, labels_target)
    return data, category_one_hot, target_one_hot

In [ ]:
data, labels_category, labels_target = load_data(destination_dir)

In [ ]:
np.savez("project_listen_dataset.npz", array1=data, array2=labels_category, array3=labels_target)